# 🐍 Day 17 · Python Basics — Dates, Web Data & Webhooks (WhatsApp Agent)

> **Goal of this notebook:** When someone messages your WhatsApp bot, Twilio sends your server a small web form. Learn dates, how to read that data and how to build a reply.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. Dates and times with `datetime`
2. What a webhook is
3. Reading form data with `urllib.parse`
4. Cleaning phone numbers and text
5. Handling text vs image messages
6. Building the reply

## 1. Dates and times

In [ ]:
from datetime import datetime, timedelta

now = datetime.now()
print(now)
print(now.strftime("%d %B %Y, %H:%M"))      # format it nicely, e.g. 17 November 2026, 10:30

tomorrow = now + timedelta(days=1)
print("Tomorrow is", tomorrow.strftime("%A"))   # the weekday name

## 2. What is a webhook?
A **webhook** is a web address on *your* server that another service calls when something happens.
```
User sends WhatsApp msg  →  Twilio  →  POST https://your-server/whatsapp/webhook  →  your Python code
```
FastAPI receives the request; our job is to read the data inside it.

## 3. Reading form data
Twilio sends the message as **form data** — text that looks like `Body=Hello&From=whatsapp%3A%2B9198...`.
`parse_qs` turns that text into a dictionary.

In [ ]:
from urllib.parse import parse_qs

raw_body = "Body=What+is+RAG%3F&From=whatsapp%3A%2B919876543210&NumMedia=0"

form = parse_qs(raw_body)
print(form)                          # every value is a list

message = form["Body"][0]            # take the first value
sender = form["From"][0]
print(message, "|", sender)

## 4. Cleaning phone numbers and text

In [ ]:
phone = sender.replace("whatsapp:", "")     # remove the prefix
print(phone)
print(phone[-4:])                            # last 4 digits (nice for logs, hides the rest)

user_text = "   hello BOT!!   "
print(user_text.strip().lower())

## 5. Text vs image messages
If `NumMedia` is more than 0 the user sent a picture, and `MediaUrl0` holds its address.

In [ ]:
def message_type(form):
    num_media = int(form.get("NumMedia", ["0"])[0])     # text -> number
    if num_media > 0:
        return "image", form["MediaUrl0"][0]
    return "text", form["Body"][0]

image_form = parse_qs("NumMedia=1&MediaUrl0=https%3A%2F%2Fexample.com%2Fcat.jpg&Body=")
print(message_type(form))
print(message_type(image_form))

## 6. 🔗 Building the reply
Each phone number gets its own **thread id**, so the bot remembers every user separately.

In [ ]:
def handle_webhook(raw_body):
    form = parse_qs(raw_body)
    kind, content = message_type(form)
    phone = form["From"][0].replace("whatsapp:", "")
    config = {"configurable": {"thread_id": phone}}       # one memory per user
    if kind == "image":
        reply = "Thanks for the image! I'll take a look. 🖼️"
    else:
        reply = f"You asked: '{content}'. (A real bot would call the LangGraph agent here.)"
    time_sent = datetime.now().strftime("%H:%M")
    return {"to": phone, "reply": reply, "time": time_sent, "config": config}

print(handle_webhook(raw_body))

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Print today's date in the format `2026-11-17` (hint: `"%Y-%m-%d"`).
2. Parse `"name=Asha&city=Goa"` with `parse_qs` and print the city.
3. Get the last 3 digits of `"+14155550123"`.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
print(datetime.now().strftime("%Y-%m-%d"))
print(parse_qs("name=Asha&city=Goa")["city"][0])
print("+14155550123"[-3:])
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Now | `datetime.now()` |
| Format a date | `now.strftime('%d %B %Y')` |
| Add days | `now + timedelta(days=1)` |
| Form data -> dict | `parse_qs(raw)['Body'][0]` |
| Text -> number | `int('3')` |
| Remove prefix | `s.replace('whatsapp:', '')` |
| Return two values | `return 'image', url` |

## ➡️ What's next?
You now know the Python used in **Day_17_Project_WhatsApp_Agent**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀